# 00 — First principles of intelligent document processing

> **Applied Labs** · **Domain**: MM · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/05_document_processing/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and familiarity with text processing
- Understanding of data structures (dicts, lists)

**What you will learn**:
- The four stages of document understanding: layout perception, content extraction, semantic typing, and validation
- Why template-based extraction fails at scale (O(n) maintenance per vendor)
- How field ambiguity makes context essential for correct extraction
- Cross-field validation as an arithmetic consistency check
- Confidence-based human-in-the-loop routing and cost tradeoffs
- The $12 B+ IDP market opportunity

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "Pillow>=10.0.0" "numpy>=1.24.0" "matplotlib>=3.7.0"

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from enum import Enum
import re, textwrap, json

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What is document understanding?

Document understanding goes far beyond OCR. It is a **four-stage pipeline**:

```
┌────────────────┐   ┌────────────────┐   ┌────────────────┐   ┌────────────────┐
│  1. Layout      │──▶│  2. Content     │──▶│  3. Semantic    │──▶│  4. Validation  │
│  Perception     │   │  Extraction     │   │  Typing         │   │                 │
└────────────────┘   └────────────────┘   └────────────────┘   └────────────────┘
 Detect regions       Read text from       Assign meaning       Check consistency
 (header, table,      each region          (invoice_number,     (math, formats,
  key-value pair)                           total, date)        required fields)
```

Consider a simple invoice: OCR gives you raw text, but you still need to
understand *where* each field is, *what* it means, and *whether* the values
are internally consistent.

In [ ]:
# The four stages of document understanding

class DocumentStage(Enum):
    """Stages in the document understanding pipeline."""
    LAYOUT_PERCEPTION = "layout_perception"
    CONTENT_EXTRACTION = "content_extraction"
    SEMANTIC_TYPING = "semantic_typing"
    VALIDATION = "validation"

@dataclass
class StageDefinition:
    """Definition of a single document understanding stage."""
    stage: DocumentStage
    description: str
    inputs: List[str]
    outputs: List[str]
    example: str

stages: List[StageDefinition] = [
    StageDefinition(
        stage=DocumentStage.LAYOUT_PERCEPTION,
        description="Detect document regions: headers, tables, key-value pairs, logos",
        inputs=["raw image / PDF page"],
        outputs=["bounding boxes", "region types"],
        example="Detect a table spanning rows 5-15 with 4 columns"
    ),
    StageDefinition(
        stage=DocumentStage.CONTENT_EXTRACTION,
        description="Read text content from each detected region",
        inputs=["image regions", "bounding boxes"],
        outputs=["text per region"],
        example="Read 'Invoice #INV-2024-001' from header region"
    ),
    StageDefinition(
        stage=DocumentStage.SEMANTIC_TYPING,
        description="Assign business meaning to each extracted value",
        inputs=["text per region", "document type"],
        outputs=["typed fields (invoice_number, date, total, etc.)"],
        example="'INV-2024-001' → field: invoice_number"
    ),
    StageDefinition(
        stage=DocumentStage.VALIDATION,
        description="Check internal consistency and business rules",
        inputs=["typed fields"],
        outputs=["validated record", "confidence scores", "error flags"],
        example="Verify line_items sum equals subtotal, tax = subtotal × rate"
    ),
]

print("Four stages of document understanding")
print("=" * 60)
for i, s in enumerate(stages, 1):
    print(f"\nStage {i}: {s.stage.value}")
    print(f"  Description : {s.description}")
    print(f"  Inputs      : {', '.join(s.inputs)}")
    print(f"  Outputs     : {', '.join(s.outputs)}")
    print(f"  Example     : {s.example}")

# ── Invoice example ──
sample_invoice: Dict[str, Any] = {
    "invoice_number": "INV-2024-001",
    "date": "2024-03-15",
    "vendor": "Acme Supplies Inc.",
    "line_items": [
        {"description": "Widget A", "qty": 10, "unit_price": 25.00, "total": 250.00},
        {"description": "Widget B", "qty": 5,  "unit_price": 50.00, "total": 250.00},
        {"description": "Service Fee", "qty": 1, "unit_price": 100.00, "total": 100.00},
    ],
    "subtotal": 600.00,
    "tax_rate": 0.08,
    "tax": 48.00,
    "total": 648.00,
}

print("\n\nSample invoice (structured output)")
print("=" * 60)
print(json.dumps(sample_invoice, indent=2))
print(f"\n✓ {len(stages)} stages defined, sample invoice has {len(sample_invoice['line_items'])} line items")

## Section 2 — Why templates fail

The traditional IDP approach uses **template matching**: define fixed pixel coordinates
for each field on each vendor's invoice format.

This works perfectly — until a vendor changes their layout, or you onboard a new
vendor. Maintenance is **O(n)** where n = number of document formats.

| Vendors | Templates needed | Annual maintenance hours |
|---------|-----------------|------------------------|
| 10      | 10              | ~50 h                  |
| 100     | 100             | ~500 h                 |
| 1,000   | 1,000           | ~5,000 h               |

VLMs eliminate this by understanding layout **semantically** rather than positionally.

In [ ]:
# Template-based extraction — works for format A, breaks on format B

@dataclass
class TemplateField:
    """A field extracted by fixed position on the document."""
    name: str
    x: int
    y: int
    width: int
    height: int

@dataclass
class DocumentTemplate:
    """Template for a specific vendor/format."""
    vendor: str
    fields: List[TemplateField]

    def extract(self, document_text_grid: Dict[Tuple[int, int], str]) -> Dict[str, str]:
        """Extract fields by looking up text at fixed positions."""
        result: Dict[str, str] = {}
        for f in self.fields:
            key = (f.x, f.y)
            result[f.name] = document_text_grid.get(key, "NOT_FOUND")
        return result

# ── Template for Vendor A (total at position 400,600) ──
template_a = DocumentTemplate(
    vendor="Acme Supplies",
    fields=[
        TemplateField("invoice_number", 350, 50, 200, 30),
        TemplateField("date", 350, 90, 150, 30),
        TemplateField("total", 400, 600, 150, 30),
    ]
)

# ── Vendor A document ──
doc_a: Dict[Tuple[int, int], str] = {
    (350, 50): "INV-2024-001",
    (350, 90): "2024-03-15",
    (400, 600): "$648.00",
}

# ── Vendor B document — different layout ──
doc_b: Dict[Tuple[int, int], str] = {
    (100, 30): "B-7892",
    (100, 60): "March 20, 2024",
    (500, 700): "$1,250.00",
}

print("Template extraction — Vendor A (matching template)")
print("=" * 55)
result_a = template_a.extract(doc_a)
for k, v in result_a.items():
    status = "✓" if v != "NOT_FOUND" else "✗"
    print(f"  {status} {k:20s} → {v}")

print("\nTemplate extraction — Vendor B (different layout)")
print("=" * 55)
result_b = template_a.extract(doc_b)
for k, v in result_b.items():
    status = "✓" if v != "NOT_FOUND" else "✗"
    print(f"  {status} {k:20s} → {v}")

n_found_a = sum(1 for v in result_a.values() if v != "NOT_FOUND")
n_found_b = sum(1 for v in result_b.values() if v != "NOT_FOUND")
print(f"\nVendor A: {n_found_a}/{len(result_a)} fields extracted")
print(f"Vendor B: {n_found_b}/{len(result_b)} fields extracted")
print("\n⚠ Template designed for A fails completely on B — O(n) maintenance per vendor")

## Section 3 — The field extraction problem

Even when text is correctly read, **ambiguity** makes extraction hard. The string
`"Total: $1,500"` could be a subtotal, a grand total, or a line-item total depending
on its **context** within the document.

VLMs solve this by considering the surrounding layout, headers, and position — just
as a human would.

In [ ]:
# Field ambiguity — the same text means different things in different contexts

@dataclass
class AmbiguousField:
    """An extracted text snippet that is ambiguous without context."""
    raw_text: str
    possible_meanings: List[str]
    correct_meaning: str
    context_needed: str

ambiguous_fields: List[AmbiguousField] = [
    AmbiguousField(
        raw_text="Total: $1,500.00",
        possible_meanings=["grand_total", "subtotal", "line_item_total"],
        correct_meaning="subtotal",
        context_needed="Appears above tax line; grand total is $1,620 below"
    ),
    AmbiguousField(
        raw_text="Date: 03/15/2024",
        possible_meanings=["invoice_date", "due_date", "ship_date"],
        correct_meaning="invoice_date",
        context_needed="Labeled 'Invoice Date' in header; due date appears separately"
    ),
    AmbiguousField(
        raw_text="Net 30",
        possible_meanings=["payment_terms", "shipping_terms", "warranty_period"],
        correct_meaning="payment_terms",
        context_needed="Appears next to 'Payment Terms:' label"
    ),
    AmbiguousField(
        raw_text="$250.00",
        possible_meanings=["unit_price", "line_total", "discount", "shipping_cost"],
        correct_meaning="line_total",
        context_needed="Rightmost column in line-item table, row with qty=10 × $25"
    ),
    AmbiguousField(
        raw_text="Acme Corp",
        possible_meanings=["vendor_name", "bill_to_company", "ship_to_company"],
        correct_meaning="vendor_name",
        context_needed="Appears in top-left logo area with vendor address below"
    ),
]

print("Field extraction ambiguity — 5 examples")
print("=" * 65)
for i, af in enumerate(ambiguous_fields, 1):
    print(f"\n{i}. Raw text: '{af.raw_text}'")
    print(f"   Possible meanings : {af.possible_meanings}")
    print(f"   Correct meaning   : {af.correct_meaning}")
    print(f"   Context needed    : {af.context_needed}")

print(f"\n✓ {len(ambiguous_fields)} ambiguous fields — context resolves every one")

## Section 4 — Cross-field validation

After extraction, **arithmetic validation** catches errors that the model may
introduce. An invoice is internally consistent when:

- Each `line_total = qty × unit_price`
- `subtotal = Σ line_totals`
- `tax = subtotal × tax_rate`
- `grand_total = subtotal + tax`

Any violation signals an extraction error that needs review.

In [ ]:
# Cross-field validator catching arithmetic errors

@dataclass
class LineItem:
    """Single line item from an invoice."""
    description: str
    qty: int
    unit_price: float
    total: float

@dataclass
class InvoiceExtraction:
    """Extracted invoice with all fields."""
    invoice_id: str
    line_items: List[LineItem]
    subtotal: float
    tax_rate: float
    tax: float
    grand_total: float

@dataclass
class ValidationResult:
    """Result of cross-field validation."""
    check: str
    expected: float
    actual: float
    passed: bool
    detail: str

def validate_invoice(inv: InvoiceExtraction) -> List[ValidationResult]:
    """Validate arithmetic consistency of an invoice extraction."""
    results: List[ValidationResult] = []

    # Check each line item
    for item in inv.line_items:
        expected_total = item.qty * item.unit_price
        results.append(ValidationResult(
            check=f"line_total({item.description})",
            expected=expected_total,
            actual=item.total,
            passed=abs(expected_total - item.total) < 0.01,
            detail=f"{item.qty} × ${item.unit_price:.2f} = ${expected_total:.2f}"
        ))

    # Subtotal check
    expected_subtotal = sum(item.total for item in inv.line_items)
    results.append(ValidationResult(
        check="subtotal",
        expected=expected_subtotal,
        actual=inv.subtotal,
        passed=abs(expected_subtotal - inv.subtotal) < 0.01,
        detail=f"Σ line totals = ${expected_subtotal:.2f}"
    ))

    # Tax check
    expected_tax = round(inv.subtotal * inv.tax_rate, 2)
    results.append(ValidationResult(
        check="tax",
        expected=expected_tax,
        actual=inv.tax,
        passed=abs(expected_tax - inv.tax) < 0.01,
        detail=f"${inv.subtotal:.2f} × {inv.tax_rate:.0%} = ${expected_tax:.2f}"
    ))

    # Grand total check
    expected_grand = inv.subtotal + inv.tax
    results.append(ValidationResult(
        check="grand_total",
        expected=expected_grand,
        actual=inv.grand_total,
        passed=abs(expected_grand - inv.grand_total) < 0.01,
        detail=f"${inv.subtotal:.2f} + ${inv.tax:.2f} = ${expected_grand:.2f}"
    ))

    return results

# ── Correct invoice ──
correct_invoice = InvoiceExtraction(
    invoice_id="INV-001",
    line_items=[
        LineItem("Widget A", 10, 25.00, 250.00),
        LineItem("Widget B", 5, 50.00, 250.00),
        LineItem("Service Fee", 1, 100.00, 100.00),
    ],
    subtotal=600.00, tax_rate=0.08, tax=48.00, grand_total=648.00
)

# ── Invoice with errors (simulating extraction mistakes) ──
error_invoice = InvoiceExtraction(
    invoice_id="INV-002-BAD",
    line_items=[
        LineItem("Widget A", 10, 25.00, 250.00),
        LineItem("Widget B", 5, 50.00, 300.00),   # ERROR: should be 250
        LineItem("Service Fee", 1, 100.00, 100.00),
    ],
    subtotal=600.00, tax_rate=0.08, tax=50.00, grand_total=700.00  # Cascading errors
)

for inv in [correct_invoice, error_invoice]:
    print(f"\nValidation: {inv.invoice_id}")
    print("=" * 60)
    results = validate_invoice(inv)
    for r in results:
        icon = "✓" if r.passed else "✗"
        print(f"  {icon} {r.check:30s} expected={r.expected:>10.2f}  actual={r.actual:>10.2f}")
        if not r.passed:
            print(f"    ↳ {r.detail}")
    n_pass = sum(1 for r in results if r.passed)
    print(f"  Result: {n_pass}/{len(results)} checks passed")

## Section 5 — Confidence and human-in-the-loop routing

Not every document needs human review. A well-calibrated confidence score enables
**tiered routing**:

| Confidence | Action | Cost per doc |
|-----------|--------|-------------|
| High (≥ 0.95) | Auto-approve | ~$0.05 |
| Medium (0.70–0.95) | Quick review | ~$0.50 |
| Low (< 0.70) | Full manual | ~$3.00 |

The goal: maximise auto-approval while keeping error rate below a target threshold.

In [ ]:
# Confidence-based routing and cost tradeoff curve

@dataclass
class RoutingDecision:
    """Routing decision for a processed document."""
    doc_id: str
    confidence: float
    route: str  # auto | review | manual
    estimated_cost: float

def route_document(doc_id: str, confidence: float) -> RoutingDecision:
    """Route a document based on extraction confidence."""
    if confidence >= 0.95:
        return RoutingDecision(doc_id, confidence, "auto", 0.05)
    elif confidence >= 0.70:
        return RoutingDecision(doc_id, confidence, "review", 0.50)
    else:
        return RoutingDecision(doc_id, confidence, "manual", 3.00)

# Simulate 200 documents with varying confidence
np.random.seed(42)
confidences: np.ndarray = np.clip(np.random.beta(8, 2, size=200), 0.3, 1.0)

decisions = [route_document(f"DOC-{i:03d}", float(c)) for i, c in enumerate(confidences)]
routes = {"auto": [], "review": [], "manual": []}
for d in decisions:
    routes[d.route].append(d)

print("Routing summary (200 documents)")
print("=" * 50)
total_cost = 0.0
for route_name in ["auto", "review", "manual"]:
    docs = routes[route_name]
    cost = sum(d.estimated_cost for d in docs)
    total_cost += cost
    print(f"  {route_name:8s}: {len(docs):4d} docs  |  cost: ${cost:>8.2f}")
print(f"  {'TOTAL':8s}: {len(decisions):4d} docs  |  cost: ${total_cost:>8.2f}")

manual_baseline = len(decisions) * 3.00
savings_pct = (1 - total_cost / manual_baseline) * 100
print(f"\n  Manual baseline: ${manual_baseline:.2f}")
print(f"  Savings: {savings_pct:.0f}%")

# ── Cost tradeoff curve ──
thresholds = np.linspace(0.5, 0.99, 50)
auto_rates: List[float] = []
error_rates: List[float] = []
costs: List[float] = []

for t in thresholds:
    auto_mask = confidences >= t
    n_auto = auto_mask.sum()
    auto_rate = n_auto / len(confidences)
    # Simulate: docs below true_conf 0.90 have 15% error rate
    true_errors = np.random.random(n_auto) > (confidences[auto_mask] + 0.05)
    err_rate = true_errors.mean() if n_auto > 0 else 0.0
    cost = n_auto * 0.05 + (len(confidences) - n_auto) * 1.50
    auto_rates.append(auto_rate)
    error_rates.append(err_rate)
    costs.append(cost)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thresholds, auto_rates, "b-", linewidth=2)
ax1.set_xlabel("Auto-approve threshold")
ax1.set_ylabel("Auto-approval rate", color="b")
ax1.axvline(x=0.95, color="r", linestyle="--", alpha=0.7, label="Default threshold (0.95)")
ax1.legend()
ax1.set_title("Auto-approval rate vs threshold")
ax1.grid(True, alpha=0.3)

ax2.plot(auto_rates, costs, "g-", linewidth=2)
ax2.set_xlabel("Auto-approval rate")
ax2.set_ylabel("Total cost ($)")
ax2.set_title("Cost vs auto-approval rate (200 docs)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 6 — Market analysis

The **Intelligent Document Processing (IDP)** market is valued at over **$12 billion**
and growing rapidly. Key players and economics:

| Metric | Value |
|--------|-------|
| IDP market size (2024) | $12 B+ |
| CAGR | ~30 % |
| Manual data entry cost | $2 – $5 per document |
| OCR + template cost | $0.50 – $1.00 per doc |
| VLM-based IDP cost | $0.05 – $0.20 per doc |
| Typical enterprise volume | 100 K – 1 M docs/year |

Key players: **Reducto**, **Unstructured.io**, **AWS Textract**, **Google Document AI**,
**Azure Document Intelligence**. The shift from template OCR to VLM-based extraction
is the current inflection point.

In [ ]:
# ── ROI model for IDP ──
volumes = [10_000, 50_000, 100_000, 500_000, 1_000_000]
manual_cost_per_doc = 3.50
template_cost_per_doc = 0.75
vlm_cost_per_doc = 0.12

print("IDP ROI Analysis — Cost per year by volume")
print("=" * 70)
print(f"{'Volume':>12s} | {'Manual':>12s} | {'Template OCR':>12s} | {'VLM-based':>12s} | {'VLM Savings':>12s}")
print("-" * 70)
for vol in volumes:
    manual = vol * manual_cost_per_doc
    template = vol * template_cost_per_doc
    vlm = vol * vlm_cost_per_doc
    savings = manual - vlm
    print(f"{vol:>12,d} | ${manual:>11,.0f} | ${template:>11,.0f} | ${vlm:>11,.0f} | ${savings:>11,.0f}")

print("\nKey insight: at 100K+ docs/year, VLM-based IDP saves $300K+/year vs manual")
print("Template OCR requires ongoing maintenance ($50K+/year for 100+ vendors)")
print("\nMarket drivers:")
for d in [
    "Regulatory compliance requiring audit trails for document processing",
    "Remote work increasing digital document volume 3×",
    "VLM accuracy now exceeding template OCR on diverse layouts",
    "API-first platforms enabling quick integration (days, not months)",
]:
    print(f"  • {d}")

## Exercises

1. **Extend the validator** — Add format validation checks to `validate_invoice`:
   verify that `invoice_id` matches pattern `INV-\d{4}-\d{3}`, that `date` is a
   valid ISO date, and that all monetary values are non-negative. Run on both
   test invoices and print results.

2. **Multi-vendor templates** — Create templates for three different vendor layouts.
   Show that a single template achieves 100 % on its target vendor but < 33 % on
   the others. Calculate the total maintenance cost at 50 vendors.

3. **Confidence simulator** — Modify the routing simulation to model a bimodal
   confidence distribution (some docs are easy, some are hard). How does the
   cost curve change compared to the beta distribution used above?

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Document understanding has four stages: layout, extraction, typing, validation |
| 2 | Template-based extraction requires O(n) maintenance per vendor format |
| 3 | Field ambiguity means context (position, headers, surrounding text) is essential |
| 4 | Cross-field arithmetic validation catches extraction errors automatically |
| 5 | Confidence-based routing cuts costs 80 %+ by auto-approving high-confidence docs |
| 6 | IDP is a $12 B+ market; VLMs are replacing template OCR as the dominant approach |

## What's Next

In **01 — Architecture**, we design the full document processing pipeline:
classification, layout analysis, VLM extraction, validation, confidence scoring,
and human-in-the-loop routing.

---

## References

1. Huang, Y. et al., *LayoutLMv3: Pre-training for Document AI with Unified Text and Image Masking*, ACM MM 2022.
2. Reducto, *Intelligent Document Processing Platform* — <https://reducto.ai/>
3. AWS, *Amazon Textract Developer Guide* — <https://docs.aws.amazon.com/textract/>
4. Grand View Research, *Intelligent Document Processing Market Size Report*, 2024.